# PO-001 — Interactive Front Door

**Title:** From Covering-Edge Descent Complexes to Functorial Presheaves and Coherence Defects  
**Subtitle:** Strict Functorial Completion, Coboundary-Curvature Duality, and the Finite Obstruction Boundary  
**Author:** Jeremy H. Carroll  
**AuthorityEffect:** `None`  
**Zenodo Record:** [10.5281/zenodo.23115851](https://doi.org/10.5281/zenodo.23115851)  

Welcome to the interactive entry point for **PO-001**. This notebook walks through the key structural results:
1. **Path A:** Strict completion of covering edges to a functorial Banach presheaf via path-independent saturated chains.
2. **Path B:** The degree-2 curvature cochain $K_{c,b,a} = r_{b,a} \circ r_{c,b} - r_{c,a}$ and the exact differential identity $d_1 d_0 = K$.
3. **Degree-3 Boundary:** Evaluating $d_2 K$ across quadruple chains and verifying Theorem 3.4.
4. **LGGR Quotient Boundary:** Demonstrating why local payload compatibility does not eliminate global curvature obstruction (Theorem 3.5).

*Disclaimer:* All calculations use exact rational arithmetic. Instance checks illustrate mechanism and do not replace formal proofs.

## 0. Setup and Import from `code/`

Start from the package root or the notebook directory. Each example asserts its expected result. The package verifier replays ordinary Python cells in fresh processes; it does not exercise Jupyter widgets or certify every manuscript claim.


In [1]:
import sys
from pathlib import Path

ROOT = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (candidate / "covering_edge_descent.md").is_file()
             and (candidate / "code/restriction_system.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start this notebook from the PO-001 package or a directory inside it")
sys.path.insert(0, str(ROOT / "code"))

from covering_edge_poset import FinitePoset, CoverEdge, ComparablePair
from restriction_system import RestrictionMap, RestrictionSystem, build_path_a_completion, check_path_a_diamond_independence
from cochain_complex import Section0Cochain, coboundary_d0, coboundary_d1
from curvature_cochain import compute_curvature_operator, compute_all_curvature_operators, is_flat_system, verify_theorem_3_2_pointwise, evaluate_d2_boundary
from payload_quotient import CurvaturePacket, PayloadPrimitiveDifference, certify_local_green_global_red
from exactsym import Matrix

print("PO-001 computational engine loaded successfully.")

PO-001 computational engine loaded successfully.


## 1. Path A: Strict Algebraic Completion

The two covering paths commute, so their common product supplies the strict completion. These exact matrix fixtures check the algebraic laws; no Banach norm or contractivity certificate is supplied.


In [2]:
poset = FinitePoset(
    elements=["0", "1", "2", "3"],
    relations=[("0", "1"), ("1", "3"), ("0", "2"), ("2", "3")],
)
dims = {"0": 2, "1": 2, "2": 2, "3": 2}

# Covering restrictions commuting around diamond
r_31 = RestrictionMap("3", "1", Matrix([[1, 1], [0, 1]]))
r_10 = RestrictionMap("1", "0", Matrix([[1, 2], [0, 1]]))
r_32 = RestrictionMap("3", "2", Matrix([[1, 2], [0, 1]]))
r_20 = RestrictionMap("2", "0", Matrix([[1, 1], [0, 1]]))

cov_sys = RestrictionSystem(poset, dims, [r_31, r_10, r_32, r_20])
is_indep, defect = check_path_a_diamond_independence(poset, cov_sys, "0", "1", "2", "3")

print(f"Diamond path-independent? {is_indep}")
print(f"Defect matrix:\n{defect}")

# Complete to full presheaf
presheaf = build_path_a_completion(poset, dims, [r_31, r_10, r_32, r_20])
print(f"Direct restriction 3 -> 0 completed:\n{presheaf.get_restriction('0', '3').matrix}")
print(f"Is complete presheaf flat (K == 0)? {is_flat_system(presheaf)}")
assert is_indep
assert defect == Matrix.zeros(2, 2)
assert presheaf.get_restriction("0", "3").matrix == Matrix([[1, 3], [0, 1]])
assert is_flat_system(presheaf)


Diamond path-independent? True
Defect matrix:
Matrix([[0, 0], [0, 0]])
Direct restriction 3 -> 0 completed:
Matrix([[1, 3], [0, 1]])
Is complete presheaf flat (K == 0)? True


## 2. Path B: Theorem 3.2 Verification ($d_1 d_0 = K$)

When comparable pairs are declared directly and do not commute, $K_{c,b,a} = r_{b,a} \circ r_{c,b} - r_{c,a} \ne 0$.
Theorem 3.2 establishes that applying $d_1 d_0$ to an arbitrary section $s$ exactly yields $K_{c,b,a}(s(c))$.

In [3]:
# Linear chain 0 < 1 < 2
poset_chain = FinitePoset(elements=["0", "1", "2"], relations=[("0", "1"), ("1", "2")])
dims_chain = {"0": 2, "1": 2, "2": 2}

r_21 = RestrictionMap("2", "1", Matrix([[1, 2], [3, 4]]))
r_10 = RestrictionMap("1", "0", Matrix([[2, 0], [1, -1]]))
defect_mat = Matrix([[1, -1], [2, 0]])
r_20 = RestrictionMap("2", "0", r_10.matrix * r_21.matrix - defect_mat)

sys_curv = RestrictionSystem(poset_chain, dims_chain, [r_21, r_10, r_20])

s = Section0Cochain(poset_chain, {
    "0": Matrix([[5], [7]]),
    "1": Matrix([[-3], [4]]),
    "2": Matrix([[2], [-1]]),
})

matches, d1_d0_val, k_val = verify_theorem_3_2_pointwise(sys_curv, s, "0", "1", "2")
print(f"Theorem 3.2 holds (d1(d0 s) == K(s(c)))? {matches}")
print(f"d1(d0 s) value:\n{d1_d0_val}")
print(f"K(s(c)) value:\n{k_val}")
assert matches
assert d1_d0_val == k_val == Matrix([[3], [4]])
assert not is_flat_system(sys_curv)


Theorem 3.2 holds (d1(d0 s) == K(s(c)))? True
d1(d0 s) value:
Matrix([[3], [4]])
K(s(c)) value:
Matrix([[3], [4]])


## 3. Theorem 3.4: Degree-3 Curvature Boundary ($d_2 K$)

On a 4-element chain $0 < 1 < 2 < 3$, Theorem 3.4 establishes:
$$d_2 K(s) = K_{c,b,a}(r_{d,c} s(d)) - K_{c,b,a}(s(c)).$$
We evaluate both expressions and verify their exact algebraic equality.

In [4]:
poset_4 = FinitePoset(elements=["0", "1", "2", "3"], relations=[("0", "1"), ("1", "2"), ("2", "3")])
dims_4 = {"0": 2, "1": 2, "2": 2, "3": 2}

r_32 = RestrictionMap("3", "2", Matrix([[2, 1], [0, 1]]))
r_21 = RestrictionMap("2", "1", Matrix([[1, 0], [1, 1]]))
r_10 = RestrictionMap("1", "0", Matrix([[3, 1], [2, 1]]))
r_31 = RestrictionMap("3", "1", Matrix([[5, 2], [1, 3]]))
r_30 = RestrictionMap("3", "0", Matrix([[10, 4], [3, 2]]))
r_20 = RestrictionMap("2", "0", Matrix([[4, 1], [3, 1]]))

sys_4 = RestrictionSystem(poset_4, dims_4, [r_32, r_21, r_10, r_31, r_30, r_20])

s_3 = Matrix([[3], [-2]])
s_2 = Matrix([[1], [4]])

d2_val, lower_diff = evaluate_d2_boundary(sys_4, "0", "1", "2", "3", s_3, s_2)
print(f"Theorem 3.4 holds (d2 K == lower curvature difference)? {d2_val == lower_diff}")
print(f"d2 K result:\n{d2_val}")
print(f"Lower difference:\n{lower_diff}")
assert d2_val == lower_diff


Theorem 3.4 holds (d2 K == lower curvature difference)? True
d2 K result:
Matrix([[0], [0]])
Lower difference:
Matrix([[0], [0]])


## 4. Theorem 3.5: Primitive Difference / LGGR Boundary

We test the Local Green, Global Red quotient boundary. A verifier identifying packets solely by local payload cannot observe differences in global curvature.

In [5]:
payload = Matrix([[42], [7]])
curv_k = Matrix([[0]])
curv_l = Matrix([[99]])

cert = certify_local_green_global_red(payload, curv_k, curv_l)
print(f"Packets payload-equivalent? {cert['payload_equivalent']}")
print(f"Local payload factors through quotient (Green)? {cert['local_payload_factors']}")
print(f"Global curvature factors through quotient (Red)? {cert['global_curvature_factors']}")
print(f"LGGR boundary certified? {cert['lggr_boundary_confirmed']}")
assert cert["payload_equivalent"]
assert cert["local_payload_factors"]
assert not cert["global_curvature_factors"]
assert cert["lggr_boundary_confirmed"]


Packets payload-equivalent? True
Local payload factors through quotient (Green)? True
Global curvature factors through quotient (Red)? False
LGGR boundary certified? True
